**Note.** The observation symbol is the binned `Damage_Score`, a manual score tied to the same grading that defines `Condition_Class`. Read the accuracy as 'condition from a graded damage score'. Splits are image-grouped (no Image_ID overlap).

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    classification_report,
    accuracy_score
)

In [2]:
STATES = ["Excellent", "Good", "Fair", "Poor"]
TARGET = "Condition_Class"

# Load datasets
train = pd.read_csv("../training_image_features_grouped.csv")
test = pd.read_csv("../testing_image_features_grouped.csv")


# Sort samples within each image
def ordered_groups(df):

    df = df.copy()

    df["_sample_num"] = pd.to_numeric(
        df["Sample_ID"].astype(str).str.extract(r"(\d+)$")[0],
        errors="coerce"
    )

    for _, group in df.groupby("Image_ID", sort=True):
        yield group.sort_values("_sample_num")

In [3]:
def viterbi_decode(
    observations,
    start_prob,
    transition_prob,
    emission_prob
):
    observations = np.asarray(observations, dtype=int)

    if len(observations) == 0:
        return np.array([], dtype=str)

    S = len(STATES)

    if np.any(observations < 0) or np.any(
        observations >= emission_prob.shape[1]
    ):
        raise ValueError("Invalid observation symbol")

    if (
        np.any(start_prob <= 0)
        or np.any(transition_prob <= 0)
        or np.any(emission_prob <= 0)
    ):
        raise ValueError("Probabilities must be positive")

    n = len(observations)

    log_start = np.log(start_prob)
    log_transition = np.log(transition_prob)
    log_emission = np.log(emission_prob)

    dp = np.full((n, S), -np.inf)
    back = np.zeros((n, S), dtype=int)

    # Initialization
    dp[0] = (
        log_start
        + log_emission[:, observations[0]]
    )

    # Recursion
    for t in range(1, n):
        scores = dp[t - 1][:, None] + log_transition

        back[t] = np.argmax(scores, axis=0)

        dp[t] = (
            np.max(scores, axis=0)
            + log_emission[:, observations[t]]
        )

    # Termination
    path = np.zeros(n, dtype=int)
    path[-1] = np.argmax(dp[-1])

    # Backtracking
    for t in range(n - 2, -1, -1):
        path[t] = back[t + 1, path[t + 1]]

    return np.array(STATES)[path]

In [4]:
damage = pd.to_numeric(
    train["Damage_Score"], errors="coerce"
)

median_damage = damage.median()
damage = damage.fillna(median_damage)

cut_points = np.unique(
    np.quantile(damage, [0.25, 0.50, 0.75])
)

n_symbols = len(cut_points) + 1


def get_observations(group):

    scores = pd.to_numeric(
        group["Damage_Score"], errors="coerce"
    ).fillna(median_damage)

    return np.digitize(scores, cut_points)


# Estimate HMM probabilities from training data
state_index = {
    state: i for i, state in enumerate(STATES)
}

S = len(STATES)
alpha = 0.5

start_prob = np.full(S, alpha)

transition_prob = np.full(
    (S, S), alpha
)

emission_prob = np.full(
    (S, n_symbols), alpha
)

for group in ordered_groups(train):

    labels = group[TARGET].astype(str).tolist()
    observations = get_observations(group)

    states = [
        state_index[label] for label in labels
    ]

    if len(states) == 0:
        continue

    # Initial probabilities
    start_prob[states[0]] += 1

    # Transition probabilities
    for a, b in zip(states[:-1], states[1:]):
        transition_prob[a, b] += 1

    # Emission probabilities
    for state, obs in zip(states, observations):
        emission_prob[state, obs] += 1


# Normalize probabilities
start_prob /= start_prob.sum()

transition_prob /= transition_prob.sum(
    axis=1, keepdims=True
)

emission_prob /= emission_prob.sum(
    axis=1, keepdims=True
)

In [5]:
y_true = []
y_pred = []

for group in ordered_groups(test):

    labels = group[TARGET].astype(str).tolist()
    observations = get_observations(group)

    predictions = viterbi_decode(
        observations,
        start_prob,
        transition_prob,
        emission_prob
    )

    y_true.extend(labels)
    y_pred.extend(predictions.tolist())


# Classification metrics
print("Viterbi Algorithm")
print("Accuracy:", accuracy_score(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    labels=STATES,
    zero_division=0
))

print("Majority-class baseline accuracy:", round(pd.Series(y_true).value_counts(normalize=True).max(), 4))


Viterbi Algorithm
Accuracy: 0.9016641452344932
              precision    recall  f1-score   support

   Excellent       0.00      0.00      0.00        32
        Good       0.95      1.00      0.97       586
        Fair       0.90      0.98      0.94      1788
        Poor       0.56      0.18      0.28       238

    accuracy                           0.90      2644
   macro avg       0.60      0.54      0.55      2644
weighted avg       0.87      0.90      0.88      2644

Majority-class baseline accuracy: 0.6762


In [6]:
import matplotlib.pyplot as plt
import numpy as np

# Convert predicted states to numeric values
state_to_num = {
    state: i for i, state in enumerate(STATES)
}

predicted_states = [
    state_to_num[state] for state in y_pred
]

# Plot state sequence
plt.figure(figsize=(10, 5))

plt.step(
    range(len(predicted_states)),
    predicted_states,
    where="mid",
    linewidth=1.5
)

plt.yticks(
    range(len(STATES)),
    STATES
)

plt.title("Viterbi Most Likely Condition State Sequence")
plt.xlabel("Test Sample Sequence")
plt.ylabel("Predicted Hidden State")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

C:\Users\Ryan John\AppData\Local\Temp\ipykernel_25164\83057191.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
